In [ ]:
import time
import requests

# NOTE: do NOT use the given client ID for your own application, it is only for educational purposes. 
# You must create your own application in the Microsoft Entra admin center and use its client ID instead.

# Get this by creating a new application in the Microsoft Entra admin center
CLIENT_ID = "e5226706-5096-431d-9516-ae48fe263401" # Pandora Launcher's OAuth client ID for this test

# Also you must request access to the api from
# https://help.minecraft.net/hc/en-us/articles/16254801392141
# for your application to be able to authenticate with Minecraft services

In [ ]:
# STEP 1: Start a temporary localhost callback server and open Microsoft OAuth login
import base64
import hashlib
import secrets
import threading
import webbrowser
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlencode, urlparse, parse_qs

AUTH_URL = "https://login.microsoftonline.com/consumers/oauth2/v2.0/authorize"
token_url = "https://login.microsoftonline.com/consumers/oauth2/v2.0/token"
scope = "XboxLive.signin offline_access"

# PKCE protects the authorization code during the localhost redirect.
code_verifier = secrets.token_urlsafe(64)
code_challenge = base64.urlsafe_b64encode(
    hashlib.sha256(code_verifier.encode("ascii")).digest()
).rstrip(b"=").decode("ascii")
oauth_state = secrets.token_urlsafe(32)

oauth_result = {}
class OAuthCallbackHandler(BaseHTTPRequestHandler):
    def send_page(self, status, page):
        body = f"<!doctype html><html><body><h2>{page}</h2></body></html>".encode()
        self.send_response(status)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        parsed = urlparse(self.path)
        if parsed.path != "/auth":
            self.send_page(404, "Page not found.")
            return

        params = parse_qs(parsed.query)
        returned_state = params.get("state", [None])[0]
        if returned_state != oauth_state:
            oauth_result["error"] = "OAuth state mismatch."
            status, page = 400, "Login failed: invalid state. You can close this tab."
        elif "error" in params:
            oauth_result["error"] = params.get("error_description", params["error"])[0]
            status, page = 400, "Microsoft sign-in was not completed. You can close this tab."
        elif "code" in params:
            oauth_result["code"] = params["code"][0]
            status, page = 200, "Microsoft sign-in received. You can close this tab and return to the app."
        else:
            oauth_result["error"] = "Authorization code missing."
            status, page = 400, "Login failed: authorization code missing."

        self.send_page(status, page)

    def log_message(self, format, *args):
        pass  # Keep the temporary callback server quiet.

# For Pandora, port 3160 is fixed as redirect URI is fixed to http://localhost:3160/auth
# If you are using your own application, you can use any available port and set the redirect URI accordingly in your app registration.
# https://github.com/Moulberry/PandoraLauncher/blob/2d83d6f22c43e804d9b544e8dc98d49d4874e3af/crates/auth/src/constants.rs#L5
server = HTTPServer(("127.0.0.1", 3160), OAuthCallbackHandler) 
server.timeout = 300
port = server.server_address[1]
redirect_uri = f"http://localhost:{port}/auth"

auth_params = {
    "client_id": CLIENT_ID,
    "response_type": "code",
    "redirect_uri": redirect_uri,
    "response_mode": "query",
    "scope": scope,
    "state": oauth_state,
    "code_challenge": code_challenge,
    "code_challenge_method": "S256",
}
login_url = f"{AUTH_URL}?{urlencode(auth_params)}"

print(f"Temporary callback server listening on {redirect_uri}")
print("Opening Microsoft sign-in in your browser...")
webbrowser.open(login_url)


In [ ]:
# STEP 2: Receive the OAuth callback, exchange the code, and obtain the refresh token
try:
    # Wait for one localhost callback; the server is temporary and handles one request.
    server.handle_request()
finally:
    server.server_close()

if "error" in oauth_result:
    raise RuntimeError(f"Microsoft OAuth failed: {oauth_result['error']}")
if "code" not in oauth_result:
    raise TimeoutError("No OAuth callback received within 5 minutes.")

token_payload = {
    "client_id": CLIENT_ID,
    "grant_type": "authorization_code",
    "code": oauth_result["code"],
    "redirect_uri": redirect_uri,
    "scope": scope,
    "code_verifier": code_verifier,
}
headers = {"Content-Type": "application/x-www-form-urlencoded"}

token_response = requests.post(token_url, data=token_payload, headers=headers, timeout=30)
token_data = token_response.json()

if token_response.status_code != 200:
    raise RuntimeError(f"Microsoft token exchange failed: {token_data}")

ms_access_token = token_data["access_token"]
refresh_token = token_data["refresh_token"]

print("[✓] Microsoft OAuth login successful.")
print("Access token and refresh token received (values hidden).")


In [ ]:
# STEP 3: Refresh the Microsoft access token using the stored refresh token.
# run this on launcher startup so the user does not need to log in again.
# always save the new refresh token returned by Microsoft, as refresh tokens usually gets rotated.

token_url = "https://login.microsoftonline.com/consumers/oauth2/v2.0/token"

response = requests.post(
    token_url,
    data={
        "client_id": CLIENT_ID,
        "grant_type": "refresh_token",
        "refresh_token": refresh_token,
        "scope": "XboxLive.signin offline_access"
    },
    headers={
        "Content-Type": "application/x-www-form-urlencoded"
    }
)

token_data = response.json()

if response.status_code == 200:
    ms_access_token = token_data["access_token"]
    refresh_token = token_data["refresh_token"]

    # Microsoft may rotate refresh tokens
    if "refresh_token" in token_data:
        refresh_token = token_data["refresh_token"]

    print("[🎉] Microsoft access token obtained!")
    print("Access Token:", ms_access_token)
    print("Refresh Token:", refresh_token) # Update the stored refresh token with the new one

else:
    print("[❌] Failed to refresh token")
    print(token_data)

    ms_access_token = None

In [ ]:
# STEP 4: Authenticate with Xbox Live
xbl_url = "https://user.auth.xboxlive.com/user/authenticate"

xbl_payload = {
    "Properties": {
        "AuthMethod": "RPS",
        "SiteName": "user.auth.xboxlive.com",
        "RpsTicket": f"d={ms_access_token}"
    },
    "RelyingParty": "http://auth.xboxlive.com",
    "TokenType": "JWT"
}

xbl_headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

xbl_response = requests.post(
    xbl_url,
    json=xbl_payload,
    headers=xbl_headers
)

xbl_data = xbl_response.json()

if xbl_response.status_code == 200:
    xbl_token = xbl_data["Token"]
    user_hash = xbl_data["DisplayClaims"]["xui"][0]["uhs"]

    print("[🎉] Xbox Live authentication successful!")
    print("User Hash:", user_hash)
    print("XBL Token:", xbl_token)

else:
    print("[❌] Xbox Live authentication failed")
    print(xbl_response.status_code)
    print(xbl_data)

In [ ]:
# STEP 5: Update XSTS Token from Minecraft Services relying party
xsts_url = "https://xsts.auth.xboxlive.com/xsts/authorize"

xsts_payload = {
    "Properties": {
        "SandboxId": "RETAIL",
        "UserTokens": [
            xbl_token
        ]
    },
    "RelyingParty": "rp://api.minecraftservices.com/",
    "TokenType": "JWT"
}

xsts_headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

xsts_response = requests.post(
    xsts_url,
    json=xsts_payload,
    headers=xsts_headers
)

xsts_data = xsts_response.json()

if xsts_response.status_code == 200:
    xsts_token = xsts_data["Token"]

    # Update xui from the XSTS response
    xui = xsts_data["DisplayClaims"]["xui"][0]
    user_hash = xui["uhs"]

    print("[🎉] XSTS authorization successful!")
    print("User Hash:", user_hash)
    print("XSTS Token:", xsts_token)

else:
    print("[❌] XSTS authorization failed")

    # common XSTS errors related to Xbox accounts, with their corresponding error codes and messages
    if "XErr" in xsts_data:
        xerr = xsts_data["XErr"]

        if xerr == 2148916233:
            print("No Xbox profile found. Sign in to xbox.com and complete Xbox account setup.")
        elif xerr == 2148916235:
            print("Xbox Live is unavailable for this account or region.")
        elif xerr == 2148916236:
            print("Adult verification is required for this Microsoft account.")
        elif xerr == 2148916237:
            print("Age verification is required for this Microsoft account.")
        elif xerr == 2148916238:
            print("This child account must be added to a Microsoft Family by an adult.")
        else:
            print(f"Unknown XSTS error: {xerr}")

    print(xsts_response.status_code)
    print(xsts_data)

    # Common Xbox account errors
    if "XErr" in xsts_data:
        print("XErr:", xsts_data["XErr"])


In [ ]:
# STEP 6: Authenticate with Minecraft
mc_auth_url = "https://api.minecraftservices.com/authentication/login_with_xbox"
identity_token = f"XBL3.0 x={user_hash};{xsts_token}"

mc_auth_payload = {
    "identityToken": identity_token
}

mc_auth_response = requests.post(
    mc_auth_url,
    json=mc_auth_payload
)

mc_auth_data = mc_auth_response.json()

if mc_auth_response.status_code == 200:
    mc_access_token = mc_auth_data["access_token"]

    print("[🎉] Minecraft authentication successful!")
    print("Minecraft Access Token:", mc_access_token)
    print(mc_auth_data)

else:
    print("[❌] Minecraft authentication failed")
    print(mc_auth_response.status_code)
    print(mc_auth_data)

In [ ]:
# STEP 7: Check if the account owns Minecraft Java Edition and fetch the profile
profile_url = "https://api.minecraftservices.com/minecraft/profile"

profile_headers = {
    "Authorization": f"Bearer {mc_access_token}",
    "Content-Type": "application/json"
}

profile_response = requests.get(profile_url, headers=profile_headers)

if profile_response.status_code == 200:
    profile_data = profile_response.json()

    username = profile_data["name"]
    uuid = profile_data["id"]

    print("[🎉] Minecraft Java Edition ownership confirmed.")
    print("Minecraft Username:", username)
    print("Minecraft UUID:", uuid)

elif profile_response.status_code == 404:
    print("[❌] This account does not own Minecraft Java Edition or does not have a Java profile.")

else:
    print("[❌] Failed to fetch Minecraft profile.")
    print(profile_response.status_code)
    print(profile_response.text)

In [9]:
# Dump into json
import json

data = {
    "username": username,
    "uuid": uuid,
    "token": mc_access_token,
    "clientid": CLIENT_ID,
    "auth_xuid": 0
}

with open("data.json", "w") as f:
    json.dump(data, f, indent=4)